# S01 — BSP from Tabular Creep Modulus Data
*Covers exam questions: Q16 (creep compliance) and Q17–Q19 (BSP with TTS)*

---

## Part 1 — Theory Recap

### From Creep Modulus to Creep Compliance (Q16)
$$J(T, t) = \frac{1}{E_c(T, t)}$$

**Units:** If $E_c$ is given in MPa, then $J$ is in MPa$^{-1}$.  
To convert to SI (1/Pa): $J_{\text{Pa}} = J_{\text{MPa}} \times 10^{-6}$.  
Strain is dimensionless: $\varepsilon = \sigma_{\text{MPa}} \times J_{\text{MPa}^{-1}}$ (consistent units).

### Time–Temperature Superposition (TTS) — Horizontal Shift
To find the effective time $t_{\text{eff}}$ at the **calculation temperature** $T_{\text{calc}}$  
for a load applied at **actual temperature** $T_{\text{actual}}$ for duration $\Delta t$:

> Find $t_{\text{eff}}$ on the $T_{\text{calc}}$ column such that:
> $$E_c(T_{\text{calc}},\, t_{\text{eff}}) = E_c(T_{\text{actual}},\, \Delta t)$$
> using **log–log interpolation** (or extrapolation if outside table range).

**Key insight:** When $T_{\text{actual}} > T_{\text{calc}}$, the material creeps faster at the actual temperature,  
so $t_{\text{eff}} > \Delta t$ — the same effect requires *more* equivalent time at the cooler reference.

### Boltzmann Superposition Principle (BSP)
$$\varepsilon(t_{\text{total}}) = \sum_i \Delta\sigma_i \cdot J\!\left(T_{\text{calc}},\; t_{\text{total,eff}} - \sum_{j < i} t_{j,\text{eff}}\right)$$

- **Causality:** any term where elapsed time $\leq 0$ contributes zero.
- For **recovery** steps ($\Delta\sigma < 0$, stress removed), the step adds a negative term.

### Parameter Table
| Symbol | Description | Unit |
|--------|-------------|------|
| $E_c(T, t)$ | Creep modulus at temperature $T$ and time $t$ | MPa |
| $J(T, t)$ | Creep compliance $= 1/E_c$ | MPa$^{-1}$ |
| $t_{\text{eff}}$ | Effective time at $T_{\text{calc}}$ for step at $T_{\text{actual}}$ | h |
| $\Delta\sigma_i$ | Stress increment at step $i$ (negative for unloading) | MPa |
| $T_{\text{calc}}$ | Calculation temperature (use this $E_c$ column for BSP) | °C |

---

## ⚠ Common Exam Pitfalls
1. **Unit confusion:** $\varepsilon = \sigma_{\text{MPa}} / E_{c,\text{MPa}}$ gives a dimensionless fraction — multiply by 100 for %.
2. **Direction of TTS shift:** Higher temperature → more creep → $t_{\text{eff}} > \Delta t$ (longer effective time at $T_{\text{calc}}$).
3. **Recovery step:** The stress increment is **negative** ($\Delta\sigma = -\sigma$); the BSP term subtracts, reducing strain.
4. **Extrapolation needed:** If $E_c(T_{\text{actual}}, \Delta t)$ is **below** the minimum value in the $T_{\text{calc}}$ column, $t_{\text{eff}}$ falls outside the table — log–log extrapolation is required.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Problem Inputs  (edit values here)
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np

# ── Creep Modulus Table ───────────────────────────────────────────────────────
# times_h: time axis shared by all temperature columns [hours]
# Ec_MPa: dict of {temperature_°C: [E_c values at each time_h]} in MPa
times_h = [1, 10, 100, 1000, 10000]

Ec_MPa = {          # PA12 example (from exam Q16–Q19)
    30: [880, 788, 683, 565, 464],
    50: [629, 525, 418, 356, 295],
    70: [400, 343, 283, 246, 209],
}

# ── Calculation Temperature ───────────────────────────────────────────────────
T_calc = 30   # °C — use this column for BSP J lookups

# ── Load History ─────────────────────────────────────────────────────────────
# List of (duration_h, temperature_°C, stress_MPa)
# Use stress_MPa = 0 for recovery (unloaded) steps.
# Q17 example: 10h load at 70°C, then 1h recovery at 30°C
load_history = [
    (10, 70, 5.0),   # loading step:  10h at 70°C, σ = 5 MPa
    ( 1, 30, 0.0),   # recovery step:  1h at 30°C, σ = 0 MPa
]
# Q18: change step 1 to (10, 50, 5.0)
# Q19: change step 2 to (10, 30, 0.0)

# ── Design limit ─────────────────────────────────────────────────────────────
STRAIN_LIMIT_PCT = 5.0   # allowable strain [%]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Helper Functions
# ═══════════════════════════════════════════════════════════════════════════════

def loglog_interp_extrap(times, Ec_col, target_Ec):
    """Find t where E_c = target_Ec via log-log interpolation or extrapolation.
    E_c must decrease monotonically with time (physically required)."""
    log_t = np.log10(np.array(times, dtype=float))
    log_E = np.log10(np.array(Ec_col, dtype=float))
    log_x = np.log10(float(target_Ec))

    # Reverse so log_E is increasing (needed by np.interp)
    log_E_asc = log_E[::-1]   # smallest E_c first  (largest t)
    log_t_asc = log_t[::-1]   # corresponding t values

    if log_x <= log_E_asc[0]:
        # Extrapolate: target < min(Ec_col) → t > max(times)
        slope = (log_t_asc[1] - log_t_asc[0]) / (log_E_asc[1] - log_E_asc[0])
        log_t_eff = log_t_asc[0] + slope * (log_x - log_E_asc[0])
    elif log_x >= log_E_asc[-1]:
        # Extrapolate: target > max(Ec_col) → t < min(times)
        slope = (log_t_asc[-1] - log_t_asc[-2]) / (log_E_asc[-1] - log_E_asc[-2])
        log_t_eff = log_t_asc[-1] + slope * (log_x - log_E_asc[-1])
    else:
        log_t_eff = np.interp(log_x, log_E_asc, log_t_asc)

    return 10.0 ** log_t_eff


def get_Ec_at(times, Ec_col, t_query):
    """Interpolate or extrapolate E_c at arbitrary time via log-log."""
    log_t = np.log10(np.array(times, dtype=float))
    log_E = np.log10(np.array(Ec_col, dtype=float))
    log_t_q = np.log10(float(t_query))

    if log_t_q >= log_t[-1]:
        # Extrapolate beyond last time point (two-point log-log slope)
        slope = (log_E[-1] - log_E[-2]) / (log_t[-1] - log_t[-2])
        return 10.0 ** (log_E[-1] + slope * (log_t_q - log_t[-1]))
    elif log_t_q <= log_t[0]:
        slope = (log_E[1] - log_E[0]) / (log_t[1] - log_t[0])
        return 10.0 ** (log_E[0] + slope * (log_t_q - log_t[0]))
    else:
        return 10.0 ** np.interp(log_t_q, log_t, log_E)


print('Helper functions defined.')

In [ ]:
# ── Q16: Creep Compliance Table  J(T,t) = 1 / E_c(T,t) ──────────────────────
print('--- Creep Compliance Table  J(T, t) = 1 / E_c(T, t) ---')
temps = sorted(Ec_MPa.keys())
header = f"  {'t [h]':<10}" + ''.join(f"  J({T}°C) [1/MPa]" for T in temps)
print(header)
print('  ' + '-' * (len(header) - 2))
for i, t in enumerate(times_h):
    row = f"  {t:<10}"
    for T in temps:
        Ec = Ec_MPa[T][i]
        J  = 1.0 / Ec
        row += f"  {J:.2e}          "
    print(row)
print()
print('  Note: to convert to SI (1/Pa) multiply each value by 1e-6')

In [ ]:
# ── Q17–Q19: BSP with TTS ────────────────────────────────────────────────────
Ec_calc = Ec_MPa[T_calc]   # T_calc column used for all J evaluations

print(f'--- BSP Calculation  (T_calc = {T_calc}°C) ---')
print()

# Step 1: build effective-time axis at T_calc
# For each history step, find t_eff: the equivalent duration at T_calc
t_eff_steps = []
sigma_steps = []   # stress at each step (for BSP increment)

sigma_prev = 0.0
for step_idx, (dt_h, T_act, sigma) in enumerate(load_history):
    delta_sigma = sigma - sigma_prev   # stress INCREMENT at start of this step
    sigma_steps.append(delta_sigma)
    sigma_prev = sigma

    if T_act == T_calc:
        t_eff = dt_h   # no TTS shift needed
        note = 'no shift (T_act = T_calc)'
    else:
        # Interpolate E_c at actual temperature and duration
        Ec_act = get_Ec_at(times_h, Ec_MPa[T_act], dt_h)
        # Horizontal shift: find t_eff at T_calc with same E_c
        t_eff = loglog_interp_extrap(times_h, Ec_calc, Ec_act)
        note = f'E_c({T_act}°C, {dt_h}h) = {Ec_act:.1f} MPa  →  t_eff at {T_calc}°C'

    t_eff_steps.append(t_eff)
    print(f'  Step {step_idx+1}: {dt_h}h at {T_act}°C  |  σ={sigma} MPa  |  t_eff = {t_eff:.2e} h  ({note})')

t_total_eff = sum(t_eff_steps)
print(f'  Total effective time at {T_calc}°C: {t_total_eff:.2e} h')
print()

# Step 2: BSP accumulation
# Δσ_i applied at cumulative time Σⱼ<ᵢ t_j_eff
# Elapsed time for term i = t_total_eff - Σⱼ<ᵢ t_j_eff
print(f'--- BSP Terms ---')
print(f"  {'Step':<6} {'Δσ [MPa]':<12} {'Elapsed_eff [h]':<20} {'E_c [MPa]':<14} {'J [1/MPa]':<14} {'ε_term [%]':<12}")
print(f"  {'----':<6} {'-'*12} {'-'*20} {'-'*14} {'-'*14} {'-'*12}")

cumulative_t_before = 0.0
eps_total = 0.0
for i, (delta_sig, t_step_eff) in enumerate(zip(sigma_steps, t_eff_steps)):
    elapsed_eff = t_total_eff - cumulative_t_before
    cumulative_t_before += t_step_eff

    if elapsed_eff <= 0 or delta_sig == 0.0:
        continue   # causality or zero-increment step

    # J at T_calc for elapsed_eff
    Ec_val = get_Ec_at(times_h, Ec_calc, elapsed_eff)
    J_val  = 1.0 / Ec_val
    eps_term = delta_sig * J_val
    eps_total += eps_term

    dt_orig, T_orig, sig_orig = load_history[i]
    print(f"  {i+1:<6} {delta_sig:<12.2f} {elapsed_eff:<20.2e} {Ec_val:<14.1f} {J_val:<14.4e} {eps_term*100:<12.4f}")

print()
print(f"  {'Total strain ε':<30}: {eps_total*100:.4f} %")

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
passed = eps_total * 100 <= STRAIN_LIMIT_PCT
print('--- VALIDATION ---')
print(f"  {'Calculated strain':<30}: {eps_total*100:.4f} %")
print(f"  {'Allowable strain':<30}: {STRAIN_LIMIT_PCT} %")
print(f"  {'Result':<30}: {'PASS' if passed else 'FAIL'}")